# 23 — Knowledge Distillation (30B MoE Teacher → 8B Dense Student)

Transfers the trained Qwen3-Coder-30B-A3B MoE model's ARO expertise into a
Qwen3-8B dense model — 3x smaller, 3x faster, same architecture family.

## Why distill?

The pipeline produces a strong 30B MoE teacher (NB18-NB21) trained on ~1,800 samples.
But 30B MoE is 16 GB and slow. An 8B dense Qwen3 model is ~4.5 GB, 3x faster, and
reliable for tool calling — but fine-tuning it directly on 1,800 samples wastes
the 30B's generalization ability.

Distillation amplifies the data: the 30B generates ~10K high-quality ARO outputs
that the 8B student trains on. The student gets the teacher's reasoning, not just
the raw training data.

## Pipeline

```
1. Generate diverse prompts (actions x patterns x complexity)
2. Run each through the trained 30B MoE teacher
3. Validate with `aro check` — keep only passing outputs
4. Merge with original training samples
5. Fine-tune the Qwen3-8B student on the combined ~12K dataset
6. Evaluate student vs teacher on the test set
```

**Input:**
- Best 30B teacher model (from NB19/21/18)
- Knowledge base (`knowledge.json`) for prompt generation
- Original dataset (`05_dataset/train.jsonl`) for merging

**Output:**
- `data/06_distill/teacher_outputs.jsonl` — validated teacher generations
- `data/06_distill/student_train.jsonl` — merged training set for student
- `models/distill/student/fused/` — the distilled Qwen3-8B model

## Setup

In [ ]:
import sys, importlib, json, random, subprocess, tempfile, time, re, shutil
from pathlib import Path
from collections import Counter

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

# ── Distillation switch ─────────────────────────────────────────────────────
# DISTILL_ENABLED gates the entire student-training track.
#
# This is enabled because STUDENT_MODEL_ID in config.py is now a non-quantized
# base (mlx-community/Qwen3-8B-bf16, 16.4 GB). LoRA training and fusion on a
# BF16 base is reliable — unlike fusing on a 4-bit base, which previously
# produced students that emitted only `!` tokens. Cell distill_07 still runs a
# post-fusion sanity check; if the fused student collapses, it is deleted and
# the notebook aborts so NB24 falls back to the 30B teacher.
#
# If training OOMs in cell distill_06, switch STUDENT_MODEL_ID to
# `mlx-community/Qwen3-8B-8bit` (8.7 GB) — its fuse path is also reliable.
# Set this to False to skip the student track entirely.
DISTILL_ENABLED = True

DISTILL_DIR = DATA_ROOT / '06_distill'
DISTILL_DIR.mkdir(parents=True, exist_ok=True)
DISTILL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

TEACHER_OUTPUTS  = DISTILL_DIR / 'teacher_outputs.jsonl'
STUDENT_TRAIN    = DISTILL_DIR / 'student_train.jsonl'
STUDENT_MLX_DIR  = DISTILL_DIR / 'mlx'
STUDENT_MLX_DIR.mkdir(parents=True, exist_ok=True)

# Generation settings
NUM_PROMPTS       = 8000     # diverse prompts (raised from 5000 once distill_03 ceiling lifted)
TEACHER_TEMP      = 0.3     # temperature for teacher generation
TEACHER_MAX_TOKENS = 1024
MAX_RETRIES       = 2       # retries per prompt on aro check failure

print(f'Teacher model: {MODEL_ID}')
print(f'Student model: {STUDENT_MODEL_ID}')
print(f'Distill dir:   {DISTILL_DIR}')
print(f'Distillation:  {"ENABLED" if DISTILL_ENABLED else "DISABLED — NB24 will use the 30B teacher"}')

if not DISTILL_ENABLED:
    print()
    print('  Stopping notebook here. Run NB24 directly to package the 30B teacher.')
    raise SystemExit('DISTILL_ENABLED=False — skipping student training (see NB24).')

if STUDENT_MODEL_ID.endswith('-4bit'):
    # Late guardrail in case someone re-enables distillation but forgets to
    # switch the student base. The post-fusion validator in distill_07 would
    # also catch a collapsed model, but failing fast here saves an hour.
    raise RuntimeError(
        f'STUDENT_MODEL_ID is 4-bit ({STUDENT_MODEL_ID}). LoRA-on-4bit-base + '
        f'fuse is fragile in MLX-LM and has produced collapsed students. Switch '
        f'config.py to mlx-community/Qwen3-8B-bf16 (16 GB) or '
        f'mlx-community/Qwen3-8B-8bit (8.7 GB) before continuing.'
    )

## Step 1 — Load teacher model

Uses the best available model from the pipeline (DPO > iterative > fine-tune > warm-start).

In [ ]:
# Find the best teacher model (same logic as NB22)
def find_best_teacher():
    import re as _re
    def _highest_fused(parent, label):
        if not parent.exists(): return None
        rounds = sorted(
            [d for d in parent.glob('round_*/fused') if (d / 'config.json').exists()],
            key=lambda p: int(_re.search(r'round_(\d+)', str(p)).group(1))
        )
        if rounds:
            best = rounds[-1]
            print(f'Found {label}: {best}')
            return str(best)
        return None

    # Priority: DPO > iterative > finetune > warm-start > base
    dpo = MODELS_DIR / 'dpo' / 'fused'
    if dpo.exists() and (dpo / 'config.json').exists():
        print(f'Using DPO teacher: {dpo}')
        return str(dpo)
    r = _highest_fused(ITERATIVE_MODELS_DIR, 'iterative')
    if r: return r
    r = _highest_fused(FINETUNE_MODELS_DIR, 'finetune')
    if r: return r
    warm = resolve_warm_adapter()
    if warm:
        print(f'Using warm-start adapter as teacher')
        return None  # will load base + adapter
    print(f'WARNING: No fine-tuned teacher found — using base model')
    return None

teacher_path = find_best_teacher()

load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()

if teacher_path:
    print(f'Loading teacher from {teacher_path}...')
    teacher_model, teacher_tok = load_fn(teacher_path)
else:
    print(f'Loading base model with warm adapter...')
    teacher_model, teacher_tok, _, _, _ = load_model(with_adapter=True)

kb = load_knowledge()
SYSTEM_PROMPT = build_system_prompt(kb)
print(f'Teacher loaded. System prompt: {len(SYSTEM_PROMPT)} chars')

## Step 2 — Generate diverse prompts

Programmatically generates prompts from the knowledge base:
- Action combinations (53 actions × prepositions × patterns)
- Feature set templates (HTTP, events, file, socket, lifecycle)
- Multi-file application scenarios
- Q&A from proposal sections
- Error diagnosis / debugging scenarios
- Plugin usage scenarios

In [ ]:
random.seed(42)

actions = kb.get('actions', [])
action_verbs = [a['verbs'][0] for a in actions if a.get('verbs')]
action_roles = {a['verbs'][0]: a.get('role', 'own') for a in actions if a.get('verbs')}

# ── Valid verb list for prompts (reduces hallucination) ────────────────────
VALID_VERBS_LIST = ', '.join(sorted(set(action_verbs)))

# ── Prompt templates ──────────────────────────────────────────────────────
# The earlier version capped debugging at 8 unique prompts (4 broken codes × 2
# templates) and explanation at 4 unique prompts. Expanded below so the dedup
# ceiling is high enough to actually use the full NUM_PROMPTS budget.

CODE_TEMPLATES = [
    # Single feature set — varied phrasings
    'Write an ARO feature set that {task}.',
    'Create an ARO feature set named "{name}" that {task}.',
    'Implement an ARO feature set that {task}. Use only valid ARO actions.',
    'Show me ARO code for a feature set that {task}.',
    'Build an ARO feature set whose business activity is to {task}.',
    'Define an ARO feature set "{name}" — its job is to {task}.',
    # Application-Start
    'Write an ARO Application-Start that {startup_task}.',
    'Create a minimal ARO application that {startup_task} and keeps running with Keepalive.',
    'Show an ARO Application-Start for an app that {startup_task}.',
    'Bootstrap an ARO application: {startup_task}, then Keepalive.',
    # Multi-feature
    'Write an ARO application with two feature sets: one that {task1} and another that {task2}.',
    'Create an ARO HTTP API with feature sets for {crud_entity}: list, create, get by ID, and delete.',
    'Build an ARO application with three feature sets: {task1}, {task2}, and {task3}.',
    'Design an ARO REST API for {crud_entity} with all five CRUD endpoints.',
    # Events
    'Write an ARO feature set that emits a {event_name} event, and a handler that processes it.',
    'Create an event-driven ARO application where {event_scenario}.',
    'Show how to emit a {event_name} event from one feature set and consume it in another.',
    'Build an ARO event handler triggered by {event_name}.',
    # Plugins / qualifiers
    'Write ARO code that uses a plugin qualifier to {qualifier_task}.',
    'Show how to call a plugin qualifier "{qualifier_name}" inside a feature set.',
    # For-each / iteration
    'Write an ARO feature set that iterates over a list of {items} and {loop_task} each one.',
    'Use a for-each loop in ARO to {loop_task} every {item_singular} in a list.',
    # Conditionals / guards
    'Write an ARO feature set with a When guard that {condition_task}.',
    'Show a feature set whose body executes only When {condition_task}.',
    # File operations
    'Write an ARO feature set that reads a {file_type} file and {file_task}.',
    'Show ARO code that watches a directory and {file_task} on each change.',
    # Repository / store files
    'Write an ARO feature set that observes the {entity_singular}-repository and reacts to changes.',
    'Show an ARO feature set that retrieves all rows from a .store-backed {entity_singular}-repository.',
    # Sockets / WebSocket
    'Write an ARO Application-Start that opens a TCP socket on port {port} and echoes incoming bytes.',
    'Show an ARO feature set that handles a new WebSocket connection and {ws_task}.',
    # Templating / output
    'Write an ARO feature set that renders a Mustache template with {entity_singular} data and returns HTML.',
    # Pipelines / data
    'Write an ARO feature set that filters {items} where {filter_predicate}, then {loop_task} the result.',
]

QA_TEMPLATES = [
    'What is the {action} action in ARO and when would you use it?',
    'Explain the difference between {action1} and {action2} in ARO.',
    'How do you {concept} in ARO? Give an example.',
    'What are the rules for {topic} in ARO?',
    'When should you use {pattern} in ARO vs {alternative}?',
    'Explain how {feature} works in ARO with a code example.',
    'In ARO, what does {action} return and how is its result bound?',
    'Walk me through how an ARO {topic} works end-to-end.',
    'What is the semantic role of the {action} action — request, own, response, or export?',
    'How does ARO\'s {topic} differ from how it works in mainstream languages?',
]

DEBUG_TEMPLATES = [
    'This ARO code has a bug. Find and fix it:\n\n```aro\n{broken_code}\n```',
    'Why does this ARO code fail `aro check`? Fix it:\n\n```aro\n{broken_code}\n```',
    'The following ARO snippet does not parse. Rewrite it so it passes `aro check`:\n\n```aro\n{broken_code}\n```',
    'A junior engineer wrote this ARO. Identify the issue and post the corrected version:\n\n```aro\n{broken_code}\n```',
    'Explain what is wrong with this ARO code and provide a working replacement:\n\n```aro\n{broken_code}\n```',
]

EXPLAIN_TEMPLATES = [
    'Explain this ARO code line by line:\n\n```aro\n{code}\n```',
    'In two short paragraphs, describe what this ARO feature set does:\n\n```aro\n{code}\n```',
    'A code reviewer asks: what does this ARO do? Answer briefly:\n\n```aro\n{code}\n```',
]

COMPLETION_TEMPLATES = [
    'Complete this ARO feature set so it passes `aro check`:\n\n```aro\n{prefix}\n```',
    'The following ARO feature set is missing its body. Fill it in:\n\n```aro\n{prefix}\n```',
]

# ── Fill-in values ────────────────────────────────────────────────────────

tasks = [
    # CRUD-style
    'retrieves a user by ID from a repository and returns it',
    'validates a request body and returns a 400 status on failure',
    'creates a new order with items from the request body',
    'updates an existing product\'s price in the product-repository',
    'deletes a session by ID from the sessions-repository',
    'lists all active subscriptions for a given user',
    'paginates results from the article-repository using page and per_page query parameters',
    # Events
    'sends a welcome email when a UserCreated event is received',
    'logs every PaymentReceived event to the audit log',
    'rejects a SessionExpired event if the user is still active',
    'fans out an OrderPlaced event to inventory, billing, and shipping handlers',
    # Computation
    'computes the total price from a list of items',
    'computes the average response time from a list of measurements',
    'computes the SHA-256 hash of an uploaded file',
    'computes a sliding-window average over the last 10 readings',
    # Logging / monitoring
    'logs each HTTP request to the console with timestamp',
    'records a Prometheus counter every time the endpoint is hit',
    # Transformations
    'transforms a list of names to uppercase',
    'normalises a phone number string into E.164 format',
    'flattens a nested list of items into a single list',
    # Filtering / set ops
    'filters a list of products where price > 100',
    'finds the intersection of two user-id lists',
    'returns the difference between yesterday\'s and today\'s active users',
    # I/O
    'reads a configuration file and starts the server',
    'writes a CSV report to disk with one row per order',
    'streams a large file line by line and emits an event per line',
    # Publish / cross-feature
    'publishes a notification event with the operation result',
    'publishes the computed total as `cart-total` for downstream feature sets',
    # External services
    'fetches weather data from an external API and returns it',
    'calls a third-party payment provider and stores the receipt',
    'queries a SQLite plugin and returns the matching rows',
    # Repository writes
    'stores a new record in the repository and emits a Created event',
    'rolls back a tentative store if validation fails',
    # HTTP request shaping
    'extracts path parameters and query parameters from the request',
    'extracts and validates a JWT bearer token from the Authorization header',
    # Auth / security
    'computes a hash of the password and stores the user',
    'rate-limits a caller by IP using a counters repository',
    # Lists / collections
    'merges two lists of items and removes duplicates',
    'sorts a list of orders by creation date descending',
    # File-system
    'watches a directory for file changes and logs them',
    'copies a file to a backup directory and emits a FileBackedUp event',
    # Sockets / WebSocket
    'accepts a WebSocket connection and echoes messages',
    'broadcasts a message to all connected WebSocket clients',
    # Date/time
    'compares two dates and returns the later one',
    'computes the next weekday after a given date',
    'generates a date range from start to end inclusive',
    # Templates
    'renders a template with user data and returns HTML',
    'renders a multi-section Markdown report from a list of metrics',
    # State machines
    'transitions an order from `placed` to `paid` using Accept',
]

startup_tasks = [
    'starts an HTTP server with the OpenAPI contract',
    'reads a config file and starts a file watcher',
    'starts a TCP socket server on port 9000',
    'starts a WebSocket server on port 8080 for chat',
    'logs the application version and starts all services',
    'starts the HTTP server and a background file monitor',
    'opens a SQLite plugin connection and starts the API',
    'configures runtime timeouts then starts the HTTP server',
    'parses CLI parameters then starts the appropriate service',
    'starts the metrics endpoint on /metrics in Prometheus format',
]

entities = ['users', 'products', 'orders', 'sessions', 'articles', 'tasks', 'events',
            'subscriptions', 'invoices', 'payments', 'inventory', 'audit-logs',
            'connections', 'measurements', 'reports', 'devices']

events = ['UserCreated', 'OrderPlaced', 'PaymentReceived', 'FileChanged', 'SessionExpired',
          'SubscriptionRenewed', 'PasswordReset', 'InvoiceIssued', 'StockDepleted',
          'ConnectionDropped', 'MeasurementRecorded', 'ReportGenerated']

ws_tasks = ['echoes the message back', 'broadcasts to all peers',
            'persists the message to the chat-repository',
            'parses a JSON command and dispatches it']

filter_predicates = ['status = "active"', 'price > 100', 'created_at > yesterday',
                     'role = "admin"', 'count > 0', 'expires_at < now()']

qualifier_names = ['Collections.pick-random', 'Collections.shuffle', 'Stats.sort',
                   'Stats.mean', 'Markdown.ToHTML', 'Hash.sha256']

concepts = [
    'emit custom events', 'use for-each loops', 'handle HTTP routes',
    'use Publish to share variables', 'create immutable transformations',
    'use repository observers', 'implement state machines with Accept',
    'configure runtime settings', 'use plugin qualifiers',
    'work with date/time ranges', 'split strings with regex',
    'use set operations on collections', 'render templates',
    'extract path parameters from a request', 'rate-limit requests',
    'observe a .store-backed repository', 'use the When guard',
    'use the Configure action', 'parse a JWT', 'group items by a field',
]

topics = [
    'variable immutability', 'feature set naming', 'Application-Start',
    'event-driven architecture', 'contract-first HTTP', 'string concatenation',
    'the Keepalive action', 'plugin qualifiers', 'store files',
    'state guards on event handlers', 'WebSocket support', 'metrics output',
    'context-aware formatting', 'repository observers', 'graceful shutdown',
    'the Compute action and qualifier-as-name syntax', 'sink syntax',
    'streaming execution', 'lazy evaluation',
]

# ── Broken-code corpus (greatly expanded) ────────────────────────────────
# Each entry is intentionally subtly broken — wrong preposition, missing
# braces, undefined symbol, invalid verb, bad qualifier, missing colon, etc.
# This is the main lever for distillation diversity in debugging/explanation
# task types.
broken_codes = [
    # Missing `the` and missing repository
    '(getUser: User API) {\n    Extract the <id> from <pathParameters>.\n    Retrieve the <user> from the <user-repository>.\n}',
    # Application-Start lacking Keepalive — server exits immediately
    '(Application-Start: My App) {\n    Start the <http-server>.\n    Return an <OK: status> for the <startup>.\n}',
    # Inline string concatenation (not allowed in ARO)
    '(handler: Events) {\n    Extract the <data> from the <event>.\n    Log <data> + " received" to the <console>.\n}',
    # Inline arithmetic in Compute (must use sink syntax)
    '(compute: Math) {\n    Compute the <total> = <price> * <quantity>.\n    Return the <total>.\n}',
    # Missing closing brace
    '(listUsers: User API) {\n    Retrieve the <users> from the <user-repository>.\n    Return an <OK: status> with <users>.',
    # Hallucinated verb "Send" used as imperative without preposition
    '(notify: Events) {\n    Send <welcome-email> <user>.\n}',
    # Invalid feature-set header (no business activity)
    '(deleteOrder) {\n    Extract the <id> from the <pathParameters: id>.\n    Return an <OK: status> for the <deletion>.\n}',
    # Undefined symbol used as object
    '(checkout: Cart) {\n    Compute the <total: sum> from the <prices>.\n    Return an <OK: status> with <basket>.\n}',
    # Wrong preposition (Extract uses `from`, not `with`)
    '(parseBody: API) {\n    Extract the <payload> with the <request: body>.\n}',
    # `Return` with two results (only one allowed)
    '(getOrder: Orders) {\n    Retrieve the <order> from the <order-repository>.\n    Return <OK> and <order>.\n}',
    # `When` guard followed by no body
    '(guard: Demo) {\n    When <user: role> is "admin".\n    Return an <OK: status> for the <ok>.\n}',
    # `For-each` over a non-collection literal
    '(loop: Demo) {\n    For each <i> in 5 do Log <i> to <console>.\n}',
    # `Publish as` missing alias
    '(share: Demo) {\n    Compute the <total: sum> from the <items>.\n    Publish as <total>.\n}',
    # Plugin qualifier called without handle namespace (deprecated form)
    '(pick: Demo) {\n    Compute the <one: pick-random> from the <items>.\n}',
    # Wrong qualifier-as-name syntax (qualifier missing)
    '(measure: Demo) {\n    Compute the <length: > from the <items>.\n}',
    # `Emit` used as action without preposition
    '(notify: Demo) {\n    Compute the <result: sum> from the <items>.\n    Emit <ResultReady>.\n}',
    # Missing dot at end of statement
    '(broken: Demo) {\n    Log "hello" to the <console>\n    Return an <OK: status> for the <noop>.\n}',
    # Two Application-Start blocks (only one allowed)
    '(Application-Start: App1) {\n    Log "one" to the <console>.\n    Return an <OK: status> for the <one>.\n}\n\n(Application-Start: App2) {\n    Log "two" to the <console>.\n    Return an <OK: status> for the <two>.\n}',
    # `Application-End` with invalid kind
    '(Application-End: Maybe) {\n    Log "bye" to the <console>.\n    Return an <OK: status> for the <bye>.\n}',
    # `Accept` used outside a state-machine context
    '(jump: Demo) {\n    Accept <state> as "paid".\n    Return an <OK: status> for the <jump>.\n}',
    # Missing `the` in front of repository
    '(list: Articles) {\n    Retrieve the <articles> from <article-repository>.\n    Return an <OK: status> with <articles>.\n}',
    # `Compute` without qualifier OR sink (no operation)
    '(noop: Demo) {\n    Compute the <result> from the <items>.\n    Return an <OK: status> with <result>.\n}',
    # Invalid result name (starts with digit)
    '(bad-name: Demo) {\n    Extract the <1st-item> from the <items>.\n    Return an <OK: status> with <1st-item>.\n}',
    # `Throw` outside a Return position
    '(boom: Demo) {\n    Throw <BadInput>.\n    Return an <OK: status> for the <boom>.\n}',
    # `Configure` with unknown setting
    '(setup: Demo) {\n    Configure the <runtime> with <unknown-setting>.\n    Return an <OK: status> for the <setup>.\n}',
    # `Render` without template
    '(view: Demo) {\n    Render the <html>.\n    Return an <OK: status> with <html>.\n}',
    # Sink syntax with mismatched parens
    '(calc: Demo) {\n    Compute the <total> as (<price> * <qty>.\n    Return an <OK: status> with <total>.\n}',
    # `Stop` action with wrong object
    '(shutdown: Demo) {\n    Stop the <coffee>.\n    Return an <OK: status> for the <shutdown>.\n}',
    # Mixed-case action verb (verbs are PascalCase only)
    '(noisy: Demo) {\n    log "hello" to the <console>.\n    Return an <OK: status> for the <noisy>.\n}',
    # `Emit` event without `as <event>` qualifier
    '(notify: Demo) {\n    Compute the <total: sum> from the <items>.\n    Emit <total>.\n}',
]

# Working ARO snippets for "explain this code" prompts (separate from broken set)
working_codes = [
    '(listUsers: User API) {\n    Retrieve the <users> from the <user-repository>.\n    Return an <OK: status> with <users>.\n}',
    '(getUser: User API) {\n    Extract the <id> from the <pathParameters: id>.\n    Retrieve the <user> from the <user-repository> where id = <id>.\n    Return an <OK: status> with <user>.\n}',
    '(Application-Start: Server) {\n    Log "Starting" to the <console>.\n    Start the <http-server> with <contract>.\n    Keepalive the <application> for the <events>.\n    Return an <OK: status> for the <startup>.\n}',
    '(handle-stock: StockDepleted Handler) {\n    Extract the <sku> from the <event: sku>.\n    Send the <reorder-email> to the <ops-team: email>.\n    Return an <OK: status> for the <notification>.\n}',
    '(filter-active: Reports) {\n    Compute the <active: filter> from the <users> where status = "active".\n    Return an <OK: status> with <active>.\n}',
]

# Partial feature sets used by the COMPLETION_TEMPLATES (header + open brace,
# missing body and close brace — model has to complete them).
partial_codes = [
    '(getProduct: Product API) {\n    Extract the <id> from the <pathParameters: id>.\n    ',
    '(Application-Start: TCP Echo) {\n    Log "Starting echo server" to the <console>.\n    ',
    '(handle-payment: PaymentReceived Handler) {\n    Extract the <amount> from the <event: amount>.\n    ',
    '(filter-orders: Orders) {\n    ',
]

# ── Optional: pull real prompts from knowledge_pairs.jsonl ───────────────
# Real human-curated questions ground the distillation set in actual ARO
# usage and reduce the synthetic-template echo problem.
KP_PATH = DATA_ROOT / '02_knowledge' / 'knowledge_pairs.jsonl'
real_qa_prompts = []
real_code_prompts = []
if KP_PATH.exists():
    with open(KP_PATH) as _f:
        for _line in _f:
            if not _line.strip():
                continue
            try:
                _rec = json.loads(_line)
                _msgs = _rec.get('messages', [])
                if len(_msgs) < 2:
                    continue
                _user = _msgs[1].get('content', '').strip()
                if not _user or len(_user) > 600:
                    continue
                _asst = _msgs[-1].get('content', '') if len(_msgs) >= 3 else ''
                _is_code = bool(re.search(r'```aro', _asst))
                if _is_code:
                    real_code_prompts.append(_user)
                else:
                    real_qa_prompts.append(_user)
            except Exception:
                pass
print(f'  Real prompts mined from knowledge_pairs.jsonl: '
      f'{len(real_code_prompts)} code, {len(real_qa_prompts)} QA')

# ── Generate prompts ──────────────────────────────────────────────────────
# Mix budget: 55% code synth, 15% real-code, 10% debugging, 5% completion,
# 5% explanation, 10% Q&A (synthetic + real). The non-synth slices break the
# template-echo dedup ceiling that capped earlier runs at ~734 unique prompts.

prompts = []

for _ in range(NUM_PROMPTS):
    roll = random.random()
    if roll < 0.55:                                  # 55% synthetic code generation
        tmpl = random.choice(CODE_TEMPLATES)
        ent = random.choice(entities)
        prompt = tmpl.format(
            task=random.choice(tasks),
            task1=random.choice(tasks),
            task2=random.choice(tasks),
            task3=random.choice(tasks),
            name=f'{random.choice(["get", "create", "update", "delete", "list", "find", "search"])}{ent.title().replace("-", "")}',
            startup_task=random.choice(startup_tasks),
            crud_entity=ent,
            entity_singular=ent[:-1] if ent.endswith('s') else ent,
            event_name=random.choice(events),
            event_scenario=f'a {random.choice(events)} triggers a notification',
            qualifier_task=f'sort a list of {random.choice(entities)}',
            qualifier_name=random.choice(qualifier_names),
            items=random.choice(entities),
            item_singular=ent[:-1] if ent.endswith('s') else ent,
            loop_task=random.choice(['logs', 'transforms', 'validates', 'stores', 'audits', 'normalises']),
            condition_task=f'checks if the {random.choice(entities)} list is empty',
            file_type=random.choice(['JSON', 'YAML', 'CSV', 'text', 'XML', 'TOML']),
            file_task=random.choice(['logs its contents', 'transforms the data', 'stores each record', 'returns its size', 'computes a checksum']),
            port=random.choice([8080, 9000, 9090, 7777]),
            ws_task=random.choice(ws_tasks),
            filter_predicate=random.choice(filter_predicates),
        )
        prompts.append(('code_generation', prompt))
    elif roll < 0.70 and real_code_prompts:          # 15% mined real code prompts
        prompts.append(('code_generation', random.choice(real_code_prompts)))
    elif roll < 0.80:                                # 10% debugging — now 30 × 5 = 150 max
        tmpl = random.choice(DEBUG_TEMPLATES)
        prompt = tmpl.format(broken_code=random.choice(broken_codes))
        prompts.append(('debugging', prompt))
    elif roll < 0.85:                                # 5% completion (new task type)
        tmpl = random.choice(COMPLETION_TEMPLATES)
        prompt = tmpl.format(prefix=random.choice(partial_codes))
        prompts.append(('code_generation', prompt))
    elif roll < 0.90:                                # 5% code explanation — now ~30 × 3 = 90 max
        tmpl = random.choice(EXPLAIN_TEMPLATES)
        prompt = tmpl.format(code=random.choice(working_codes + broken_codes))
        prompts.append(('code_explanation', prompt))
    elif roll < 0.97:                                # 7% synthetic Q&A
        tmpl = random.choice(QA_TEMPLATES)
        prompt = tmpl.format(
            action=random.choice(action_verbs),
            action1=random.choice(action_verbs),
            action2=random.choice(action_verbs),
            concept=random.choice(concepts),
            topic=random.choice(topics),
            pattern=random.choice(['Publish', 'Emit', 'Store', 'For-each', 'Accept', 'Configure']),
            alternative=random.choice(['Return', 'Log', 'Compute', 'Extract', 'Retrieve']),
            feature=random.choice(concepts),
        )
        prompts.append(('syntax_qa', prompt))
    elif real_qa_prompts:                            # 3% mined real Q&A
        prompts.append(('syntax_qa', random.choice(real_qa_prompts)))
    else:
        # Fallback if no real Q&A available: synthetic
        tmpl = random.choice(QA_TEMPLATES)
        prompts.append(('syntax_qa', tmpl.format(
            action=random.choice(action_verbs),
            action1=random.choice(action_verbs),
            action2=random.choice(action_verbs),
            concept=random.choice(concepts),
            topic=random.choice(topics),
            pattern='Publish', alternative='Return',
            feature=random.choice(concepts),
        )))

# Deduplicate
seen = set()
unique_prompts = []
for task_type, p in prompts:
    key = p[:200]
    if key not in seen:
        seen.add(key)
        unique_prompts.append((task_type, p))

random.shuffle(unique_prompts)
print(f'\nGenerated {len(unique_prompts)} unique prompts from {NUM_PROMPTS} samples')
print('Distribution:', dict(Counter(t for t, _ in unique_prompts)))
print()
print('Theoretical unique-prompt ceiling (per task type):')
print(f'  debugging        ~ {len(broken_codes)} codes × {len(DEBUG_TEMPLATES)} templates = '
      f'{len(broken_codes)*len(DEBUG_TEMPLATES)}')
print(f'  code_explanation ~ {len(working_codes)+len(broken_codes)} codes × {len(EXPLAIN_TEMPLATES)} templates = '
      f'{(len(working_codes)+len(broken_codes))*len(EXPLAIN_TEMPLATES)}')
print(f'  code_generation  + {len(real_code_prompts)} real prompts mined from knowledge_pairs')
print(f'  syntax_qa        + {len(real_qa_prompts)} real prompts mined from knowledge_pairs')

## Step 3 — Run teacher and validate

For each prompt, the teacher generates a response. Code outputs are validated
with `aro check`. Failed outputs are retried up to `MAX_RETRIES` times at
higher temperature. Only validated outputs are kept.

In [ ]:
# Append valid verb list to system prompt for teacher generation
# This dramatically reduces hallucinated verbs like "Return", "Compute", "Split"
TEACHER_SYSTEM_PROMPT = SYSTEM_PROMPT + f"""

IMPORTANT: Use ONLY these valid ARO action verbs: {VALID_VERBS_LIST}
Do NOT invent new verbs. If unsure, use Extract, Compute, Return, Log, or Store."""

# ── Multi-temperature teacher sampling ────────────────────────────────────
# Wider temperature spread for diversity. Each prompt is generated at every
# temperature in TEACHER_TEMPS, and EVERY output that passes aro check is
# kept. This typically 2-3x's the teacher data per prompt vs the old
# break-on-first-success retry loop.
#
# - Code outputs: kept if `aro check` passes (deduped by output text so we
#   do not store identical samples across temperatures)
# - Q&A outputs: kept if non-empty (deduped by first 300 chars)
TEACHER_TEMPS = [0.3, 0.6, 0.9, 1.2]   # was a single 0.3 with same-temp retries
TEACHER_TOP_P = 0.95                   # slightly wider than 0.9 to match the wider temp range


def generate_teacher(prompt, temperature, max_tokens=TEACHER_MAX_TOKENS):
    messages = [
        {'role': 'system', 'content': TEACHER_SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text = teacher_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = teacher_tok.encode(text)
    reply = generate_fn(
        teacher_model, teacher_tok,
        prompt=tokens,
        max_tokens=max_tokens,
        sampler=make_sampler_fn(temp=temperature, top_p=TEACHER_TOP_P),
        verbose=False,
    )
    return reply.strip()


def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


def aro_check(code):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0
    except Exception:
        return False


# ── Generate and validate ─────────────────────────────────────────────────
teacher_samples = []
passed, failed_prompts, qa_kept = 0, 0, 0

# Resume from existing outputs. Composite key (prompt[:200], temperature) so
# the multi-temp sweep can resume per (prompt × temp) pair without redoing
# work. Also dedup by output fingerprint so identical outputs from different
# temps are not stored twice.
existing_keys = set()                   # (prompt[:200], temperature) pairs
existing_outputs_by_prompt = {}         # prompt[:200] -> set of output[:300]
if TEACHER_OUTPUTS.exists():
    with open(TEACHER_OUTPUTS) as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                _pkey = rec.get('prompt', '')[:200]
                existing_keys.add((_pkey, rec.get('temperature')))
                existing_outputs_by_prompt.setdefault(_pkey, set()).add(rec.get('output', '')[:300])
                teacher_samples.append(rec)
    print(f'Resuming: {len(teacher_samples)} samples from '
          f'{len(existing_outputs_by_prompt)} prompts already saved')

t0 = time.time()


def _save_incremental():
    with open(TEACHER_OUTPUTS, 'w') as f:
        for s in teacher_samples:
            f.write(json.dumps(s) + '\n')


for i, (task_type, prompt) in enumerate(unique_prompts):
    pkey = prompt[:200]
    seen_outputs = existing_outputs_by_prompt.setdefault(pkey, set())
    prompt_kept_any = bool(seen_outputs)

    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        rate = (passed + qa_kept) / max(1, elapsed) * 3600
        print(f'  [{i+1}/{len(unique_prompts)}] passed={passed} qa={qa_kept} '
              f'failed_prompts={failed_prompts} ({rate:.0f}/hr · {len(teacher_samples)} kept)')

    for temp in TEACHER_TEMPS:
        if (pkey, temp) in existing_keys:
            # Already attempted this (prompt, temp) earlier — preserve resume
            # progress and do not recount.
            continue

        try:
            output = generate_teacher(prompt, temperature=temp)
        except Exception as e:
            print(f'    [gen error temp={temp}] {e}')
            existing_keys.add((pkey, temp))
            continue

        out_key = output[:300]
        if out_key in seen_outputs:
            # Identical output sampled at a different temperature — drop.
            existing_keys.add((pkey, temp))
            continue

        kept = False
        if task_type in ('syntax_qa', 'code_explanation'):
            if len(output) > 30:
                teacher_samples.append({
                    'task_type': task_type,
                    'prompt': prompt,
                    'output': output,
                    'temperature': temp,
                    'source': 'distill_teacher',
                    'valid': True,
                })
                qa_kept += 1
                kept = True
        else:
            blocks = extract_aro_blocks(output)
            if blocks and aro_check('\n\n'.join(blocks)):
                teacher_samples.append({
                    'task_type': task_type,
                    'prompt': prompt,
                    'output': output,
                    'temperature': temp,
                    'source': 'distill_teacher',
                    'valid': True,
                })
                passed += 1
                kept = True

        if kept:
            prompt_kept_any = True
            seen_outputs.add(out_key)
        existing_keys.add((pkey, temp))

        # Save incrementally every 50 kept samples
        if len(teacher_samples) % 50 == 0:
            _save_incremental()

    if not prompt_kept_any:
        failed_prompts += 1

# Final save
_save_incremental()

elapsed = time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min')
print(f'  Teacher outputs:    {len(teacher_samples):,}')
print(f'  Code passed:        {passed:,}')
print(f'  Q&A kept:           {qa_kept:,}')
print(f'  Prompts all-failed: {failed_prompts:,}')
print(f'  Outputs per prompt: {len(teacher_samples) / max(1, len(unique_prompts)):.2f}')
print(f'  Temps tried:        {TEACHER_TEMPS}')

## Step 4 — Merge with original training data

Combines teacher outputs with the original 1,800 training samples from NB17.
The original samples are high-quality human-curated data — they anchor the
student to the ground truth while the teacher outputs expand coverage.

In [ ]:
import csv

# ── Minimum code fraction in student training set ────────────────────────
MIN_CODE_FRACTION = 0.50  # at least 50% of training data must be code-producing tasks

# ── Max sequence length — must match --max-seq-length in cell distill_06 ─
# The 11K-char SYSTEM_PROMPT alone is ~2.7K tokens, so 2048 is too tight and
# would drop every sample. Longest observed total in prior runs: 3278 tokens.
# 4096 gives margin while still bounding memory.
STUDENT_MAX_SEQ_LEN = 4096

# Load original training data
original_train = []
orig_path = DATA_ROOT / '05_dataset' / 'train.jsonl'
if orig_path.exists():
    with open(orig_path) as f:
        for line in f:
            if line.strip():
                original_train.append(json.loads(line))
    print(f'Original training samples: {len(original_train)}')
else:
    print(f'WARNING: {orig_path} not found -- run NB17 first')

# Convert teacher outputs to ChatML format
teacher_chatml = []
for s in teacher_samples:
    teacher_chatml.append({
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': s['prompt']},
            {'role': 'assistant', 'content': s['output']},
        ],
        'task_type': s['task_type'],
        'source': 'distill_teacher',
    })

# Merge: original first (higher weight), then teacher
merged = original_train + teacher_chatml

# Deduplicate by user message prefix
seen = set()
deduped = []
for s in merged:
    key = s['messages'][1]['content'][:300]
    if key not in seen:
        seen.add(key)
        deduped.append(s)

# ── Filter samples exceeding the student's max sequence length ───────────
# teacher_tok is still loaded here; the Qwen3 family shares the same tokenizer
# vocabulary, so token counts match the Qwen3-8B student exactly.
def _seq_len(messages):
    text = teacher_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return len(teacher_tok.encode(text))

before_filter = len(deduped)
deduped = [s for s in deduped if _seq_len(s['messages']) <= STUDENT_MAX_SEQ_LEN]
dropped = before_filter - len(deduped)
print(f'Length filter: dropped {dropped} samples > {STUDENT_MAX_SEQ_LEN} tokens '
      f'(kept {len(deduped)} / {before_filter})')

# ── Complete-program filter ───────────────────────────────────────────────
# Drop code_generation/debugging samples whose assistant message lacks a
# feature-set header. Fragments dilute the structural signal and are the
# main reason the trained student fails to wrap ARO in `(name: activity) { }`.
deduped, _frag_dropped = filter_complete_program_samples(deduped)
print(f'Complete-program filter: dropped {_frag_dropped} fragment-only code samples '
      f'(kept {len(deduped)})')

# ── Enforce minimum code fraction ────────────────────────────────────────
# Classify each sample as "code-producing" or "qa"
CODE_TASK_TYPES = {'code_generation', 'debugging', 'correction', 'fim', 'full_application', 'translation'}

def _is_code_sample(s):
    tt = s.get('task_type', '')
    if tt in CODE_TASK_TYPES:
        return True
    # Check if assistant response contains ARO code blocks
    asst = s['messages'][-1]['content'] if s.get('messages') else ''
    return bool(re.findall(r'```aro\n.*?```', asst, re.DOTALL))

code_samples = [s for s in deduped if _is_code_sample(s)]
qa_samples = [s for s in deduped if not _is_code_sample(s)]

code_frac = len(code_samples) / max(1, len(deduped))
print(f'\nCode/QA split before balancing: {len(code_samples)} code ({code_frac:.0%}) / {len(qa_samples)} QA')

if code_frac < MIN_CODE_FRACTION:
    # Up-weight code samples by duplicating them to reach target fraction
    # target: code / (code + qa) >= MIN_CODE_FRACTION
    # code * k / (code * k + qa) >= MIN_CODE_FRACTION
    # k >= MIN_CODE_FRACTION * qa / (code * (1 - MIN_CODE_FRACTION))
    needed_code = int(MIN_CODE_FRACTION * len(qa_samples) / (1 - MIN_CODE_FRACTION))
    duplicates_needed = needed_code - len(code_samples)
    if duplicates_needed > 0 and code_samples:
        extra = []
        for i in range(duplicates_needed):
            extra.append(code_samples[i % len(code_samples)])
        deduped = code_samples + extra + qa_samples
        print(f'Up-weighted code: duplicated {duplicates_needed} code samples')
        print(f'New code fraction: {(len(code_samples) + duplicates_needed)} / {len(deduped)} = '
              f'{(len(code_samples) + duplicates_needed) / len(deduped):.0%}')
else:
    print(f'Code fraction {code_frac:.0%} >= {MIN_CODE_FRACTION:.0%} target -- no rebalancing needed')

random.seed(42)
random.shuffle(deduped)

# Split: 95% train, 5% valid
n = len(deduped)
split = int(n * 0.95)
train = deduped[:split]
valid = deduped[split:]

# Save full format
with open(STUDENT_TRAIN, 'w') as f:
    for s in train:
        f.write(json.dumps(s) + '\n')

# Save MLX format (messages only)
with open(STUDENT_MLX_DIR / 'train.jsonl', 'w') as f:
    for s in train:
        f.write(json.dumps({'messages': s['messages']}) + '\n')
with open(STUDENT_MLX_DIR / 'valid.jsonl', 'w') as f:
    for s in valid:
        f.write(json.dumps({'messages': s['messages']}) + '\n')

print(f'\nMerged dataset:')
print(f'  Original:           {len(original_train)}')
print(f'  Teacher:            {len(teacher_chatml)}')
print(f'  After dedup:        {before_filter}')
print(f'  After length filter:{before_filter - dropped}')
print(f'  Train:              {len(train)}')
print(f'  Valid:              {len(valid)}')
print(f'\nTask distribution:')
dist = Counter(s.get('task_type', s.get('source', '?')) for s in train)
for k, v in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {k:25s}: {v}')

# ── CSV export ───────────────────────────────────────────────────────────
_run_dir = Path('.') / 'run' / (__import__('datetime').date.today().isoformat())
_run_dir.mkdir(parents=True, exist_ok=True)
csv_path = _run_dir / '21_distillation.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['prompt', 'task_type', 'source', 'has_code', 'aro_check_pass', 'kept_for_student'])
    for s in teacher_samples:
        has_code = bool(extract_aro_blocks(s['output']))
        writer.writerow([
            s['prompt'][:200],
            s['task_type'],
            'teacher',
            has_code,
            s.get('valid', False) and has_code,
            True,
        ])
print(f'CSV saved: {csv_path}')

## Step 5 — Fine-tune student model

LoRA fine-tune the 8B student on the merged dataset.
Same hyperparameters as NB18, adjusted for the larger dataset.

In [ ]:
# Free teacher model memory
del teacher_model
import gc; gc.collect()
print('Teacher model freed.')

STUDENT_ADAPTER = DISTILL_MODELS_DIR / 'student' / 'adapter'
STUDENT_FUSED   = DISTILL_MODELS_DIR / 'student' / 'fused'
STUDENT_ADAPTER.mkdir(parents=True, exist_ok=True)

num_iters = min(3000, len(train) * 2)  # round-2 audit: was 1200, leaves data on floor  # ~2 epochs

cmd = [
    sys.executable, '-m', 'mlx_lm', 'lora',
    '--model',                   STUDENT_MODEL_ID,
    '--data',                    str(STUDENT_MLX_DIR),
    '--adapter-path',            str(STUDENT_ADAPTER),
    '--train',
    '--mask-prompt',
    '--batch-size',              '1',
    '--grad-accumulation-steps', '16',
    '--num-layers',              '16',  # round-2 audit: was 8 — close 30B→8B distillation gap
    '--learning-rate',           '2e-5',
    '--iters',                   str(num_iters),
    '--steps-per-report',        '50',
    '--steps-per-eval',          '100',
    '--save-every',              '200',
    '--max-seq-length',          str(STUDENT_MAX_SEQ_LEN),
]

print(f'Training student ({STUDENT_MODEL_ID}) on {len(train)} samples for {num_iters} iterations...')
print(f'Command: {" ".join(cmd)}\n')

result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    print(f'Training failed with exit code {result.returncode}')
else:
    print(f'\nTraining complete. Adapter: {STUDENT_ADAPTER}')

## Step 6 — Fuse student adapter

In [ ]:
if not (STUDENT_FUSED / 'config.json').exists():
    print(f'Fusing adapter into {STUDENT_FUSED}...')
    cmd = [
        sys.executable, '-m', 'mlx_lm', 'fuse',
        '--model',        STUDENT_MODEL_ID,
        '--adapter-path', str(STUDENT_ADAPTER),
        '--save-path',    str(STUDENT_FUSED),
    ]
    subprocess.run(cmd, check=True)
    print('Fused.')
else:
    print(f'Student fused model already exists: {STUDENT_FUSED}')


# ── Post-fusion validation ───────────────────────────────────────────────
# LoRA-on-quantized-4bit-base + fuse can produce broken weights that emit
# only one token (e.g. `!`). Catch this here so NB24 doesn't ship a corrupt
# model. We delete the fused dir on failure so NB23's selector falls through
# to the 30B teacher.
print('\nValidating fused student...')
_validation_prompts = [
    'Write an ARO feature set that returns an OK status.',
    'What does the Extract action do in ARO?',
    'Write a minimal ARO Application-Start.',
]

_val_model, _val_tok = load_fn(str(STUDENT_FUSED))
_collapsed = 0
_replies = []
for _p in _validation_prompts:
    _msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': _p},
    ]
    _txt = _val_tok.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
    _toks = _val_tok.encode(_txt)
    _reply = generate_fn(
        _val_model, _val_tok,
        prompt=_toks,
        max_tokens=80,
        sampler=make_sampler_fn(temp=0.2),
        verbose=False,
    ).strip()
    _replies.append(_reply)
    # Token-collapse heuristic: ≥80% of the reply is a single repeated character
    # (after stripping whitespace), or the reply is empty.
    _stripped = re.sub(r'\s+', '', _reply)
    if not _stripped:
        _collapsed += 1
        print(f'  [EMPTY]    {_p[:50]}')
    elif len(set(_stripped)) <= 2 and len(_stripped) >= 20:
        _collapsed += 1
        print(f'  [COLLAPSE] {_p[:50]} -> {_reply[:40]!r}')
    else:
        print(f'  [OK]       {_p[:50]} -> {_reply[:60]!r}')

del _val_model
import gc as _gc; _gc.collect()

if _collapsed >= 2:
    # Majority of validation prompts produced gibberish — model is broken.
    print(f'\n  FAIL: {_collapsed}/{len(_validation_prompts)} prompts collapsed.')
    print(f'  Removing broken fused student at {STUDENT_FUSED} so NB24 falls')
    print(f'  back to the 30B teacher (DPO/iterative/finetune).')
    shutil.rmtree(STUDENT_FUSED, ignore_errors=True)
    raise RuntimeError(
        'Post-fusion validation failed: the fused student emits collapsed '
        'output. This is a known failure mode of LoRA-on-4bit-base fusion in '
        'MLX-LM. Switch STUDENT_MODEL_ID in config.py to a non-quantized base '
        '(e.g. mlx-community/Qwen3-8B) or set DISTILL_ENABLED=False in cell '
        'distill_01 to skip distillation entirely.'
    )
print(f'\n  Validation passed: {len(_validation_prompts) - _collapsed}/{len(_validation_prompts)} prompts produced sensible output.')

## Step 7 — Evaluate student vs teacher

Run the test set through both models and compare syntax pass rate + output quality.

In [ ]:
# Load test set
test_path = DATA_ROOT / '05_dataset' / 'test.jsonl'
test_samples = []
if test_path.exists():
    with open(test_path) as f:
        for line in f:
            if line.strip():
                test_samples.append(json.loads(line))
print(f'Test samples: {len(test_samples)}')

# Load student model
print(f'Loading student from {STUDENT_FUSED}...')
student_model, student_tok = load_fn(str(STUDENT_FUSED))
print('Student loaded.')


# ── Fence-tolerant extractor ─────────────────────────────────────────────
# The original extract_aro_blocks (defined in cell distill_04) requires a
# closing ``` fence. With max_tokens too low or a fence-forgetting student,
# valid ARO code was being scored as a fail. This extractor:
#   1. Accepts unclosed ```aro fences (use rest of the buffer)
#   2. Falls back to the whole reply if it parses as bare ARO
def extract_aro_robust(text):
    text = text.strip()
    # Closed ```aro ... ``` blocks
    closed = [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]
    if closed:
        return closed
    # Unclosed ```aro fence (reply was truncated by max_tokens)
    m = re.search(r'```aro\n(.*)$', text, re.DOTALL)
    if m and m.group(1).strip():
        return [m.group(1).strip()]
    # Bare ARO: starts with a feature-set header `(...: ...) {`
    if re.search(r'\(\s*[\w\- ]+\s*:\s*[\w\- ]+\s*\)\s*\{', text):
        return [text]
    return []


# ── Eval ────────────────────────────────────────────────────────────────
EVAL_MAX_TOKENS = 1536  # was 512 — assistant responses can run 600-1500 tokens

def evaluate_model(model, tokenizer, label, dump_path=None):
    code_pass, code_fail, qa_ok, qa_thin = 0, 0, 0, 0
    dump = []
    for s in test_samples:
        prompt = s['messages'][1]['content']
        task = s.get('task_type', '')

        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        tokens = tokenizer.encode(text)
        reply = generate_fn(
            model, tokenizer,
            prompt=tokens,
            max_tokens=EVAL_MAX_TOKENS,
            sampler=make_sampler_fn(temp=0.1),
            verbose=False,
        ).strip()

        passed = None
        if task in ('code_generation', 'debugging', 'fim'):
            blocks = extract_aro_robust(reply)
            if blocks and aro_check('\n\n'.join(blocks)):
                code_pass += 1
                passed = True
            else:
                code_fail += 1
                passed = False
        else:
            text_only = re.sub(r'```.*?```', '', reply, flags=re.DOTALL).strip()
            if len(text_only) > 30:
                qa_ok += 1
                passed = True
            else:
                qa_thin += 1
                passed = False

        dump.append({
            'task_type': task,
            'prompt': prompt[:200],
            'reply': reply,
            'passed': passed,
        })

    if dump_path is not None:
        with open(dump_path, 'w') as f:
            for r in dump:
                f.write(json.dumps(r) + '\n')

    total_code = code_pass + code_fail
    total_qa = qa_ok + qa_thin
    print(f'\n{label}:')
    print(f'  Code:  {code_pass}/{total_code} pass ({100*code_pass/max(1,total_code):.0f}%)')
    print(f'  Q&A:   {qa_ok}/{total_qa} answered ({100*qa_ok/max(1,total_qa):.0f}%)')
    return {'code_pass': code_pass, 'code_total': total_code, 'qa_ok': qa_ok, 'qa_total': total_qa}


student_metrics = evaluate_model(
    student_model, student_tok, 'Student (8B distilled)',
    dump_path=DISTILL_DIR / 'eval_replies.jsonl',
)

# Optionally evaluate teacher too (requires reloading — skip if memory constrained)
# teacher_metrics = evaluate_model(teacher_model, teacher_tok, 'Teacher (30B)')

# Save metrics
metrics = {
    'student_model': STUDENT_MODEL_ID,
    'teacher_model': str(teacher_path or MODEL_ID),
    'teacher_outputs': len(teacher_samples),
    'student_train_samples': len(train),
    'eval_max_tokens': EVAL_MAX_TOKENS,
    'student_metrics': student_metrics,
}
with open(DISTILL_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nMetrics saved to {DISTILL_DIR / "metrics.json"}')
print(f'Raw replies saved to {DISTILL_DIR / "eval_replies.jsonl"} (inspect on failure)')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9', 'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '21_distillation.png'

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 5))

# ── Panel 1: Data amplification waterfall ─────────────────────────────
_wf_labels = ['Original\ntrain', 'Teacher\noutputs', 'Merged\n(dedup)', 'Student\ntrain']
_wf_values = [len(original_train), len(teacher_samples), len(deduped), len(train)]
_wf_colors = ['#3498db', '#2ecc71', '#9b59b6', '#e67e22']
bars = ax1.bar(_wf_labels, _wf_values, color=_wf_colors, edgecolor='white', width=0.55)
ax1.bar_label(bars, fmt='{:,.0f}', padding=4, fontsize=10, fontweight='bold')
ax1.set_ylabel('Samples')
ax1.set_title('Data Amplification', fontsize=12, fontweight='bold')
ax1.set_ylim(0, max(_wf_values) * 1.3 if _wf_values and max(_wf_values) > 0 else 10)
ax1.grid(axis='y', alpha=0.3)

# ── Panel 2: Teacher generation pass/fail ────────────────────────────
# Count teacher generation results from variables available
_t_code = sum(1 for s in teacher_samples if s.get('task_type') in ('code_generation', 'debugging'))
_t_qa = sum(1 for s in teacher_samples if s.get('task_type') not in ('code_generation', 'debugging'))
if _t_code or _t_qa:
    _pie_vals = [_t_code, _t_qa]
    _pie_labels = [f'Code\n({_t_code})', f'Q&A\n({_t_qa})']
    _pie_colors = ['#3498db', '#2ecc71']
    wedges, _, autotexts = ax2.pie(
        _pie_vals, labels=_pie_labels, colors=_pie_colors,
        autopct='%1.0f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    )
    for t in autotexts: t.set_fontsize(10); t.set_fontweight('bold')
    ax2.add_artist(plt.Circle((0, 0), 0.5, fc='white'))
    ax2.text(0, 0, f'{len(teacher_samples)}\ntotal', ha='center', va='center',
             fontsize=12, fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'no teacher data', ha='center', va='center', transform=ax2.transAxes)
ax2.set_title('Teacher Output Composition', fontsize=12, fontweight='bold')

# ── Panel 3: Student eval metrics ─────────────────────────────────────
if student_metrics:
    cp = student_metrics['code_pass']
    ct = student_metrics['code_total']
    qo = student_metrics['qa_ok']
    qt = student_metrics['qa_total']
    _eval_labels = ['Code\nsyntax', 'Q&A\nanswered']
    _eval_vals = [100 * cp / max(1, ct), 100 * qo / max(1, qt)]
    _eval_colors = ['#2ecc71', '#3498db']
    bars3 = ax3.bar(_eval_labels, _eval_vals, color=_eval_colors, edgecolor='white', width=0.4)
    ax3.bar_label(bars3, fmt='%.0f%%', padding=4, fontsize=12, fontweight='bold')
    ax3.set_ylim(0, 110)
    ax3.set_ylabel('Rate (%)')
    # Annotate raw counts
    ax3.text(0, -8, f'{cp}/{ct}', ha='center', fontsize=9, color='#666')
    ax3.text(1, -8, f'{qo}/{qt}', ha='center', fontsize=9, color='#666')
else:
    ax3.text(0.5, 0.5, 'no eval data', ha='center', va='center', transform=ax3.transAxes)
ax3.set_title('Student Evaluation', fontsize=12, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

fig.suptitle(
    f'Distillation — {STUDENT_MODEL_ID}  ·  {len(train)} train samples',
    fontsize=13, fontweight='bold', y=1.02,
)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')


## Summary

In [ ]:
print('=' * 65)
print('notebook 23 — DISTILLATION SUMMARY')
print('=' * 65)
print(f'\nTeacher: {teacher_path or MODEL_ID}')
print(f'Student: {STUDENT_MODEL_ID}')
print(f'\nData amplification:')
print(f'  Original samples:   {len(original_train):>6,}')
print(f'  Teacher outputs:    {len(teacher_samples):>6,}')
print(f'  Merged (deduped):   {len(deduped):>6,}')
print(f'  Student train set:  {len(train):>6,}')
print(f'\nStudent model: {STUDENT_FUSED}')
if student_metrics:
    cp = student_metrics['code_pass']
    ct = student_metrics['code_total']
    print(f'  Syntax pass rate: {cp}/{ct} ({100*cp/max(1,ct):.0f}%)')
print(f'\nNext: run NB24 to package the student model for distribution.')
print(f'  The student model at {STUDENT_FUSED}')
print(f'  will be picked up as the best available model.')
print('=' * 65)